# Transformer로 한국어-영어 번역기 구현

이 노트북은 **Transformer 모델**을 PyTorch로 직접 만들고,
실제 한국어-영어 문장 쌍을 학습시켜 번역기를 완성하는 과정을 담고 있습니다.

## 진행 구조

| 단계 | 내용 |
|:---:|---|
| Step 1 | 환경 준비 및 라이브러리 |
| Step 2 | 데이터 정제 및 SentencePiece 토큰화 |
| Step 3 | 모델 아키텍처 구현 (PE → MHA → FFN → Encoder/Decoder → Transformer) |
| Step 4 | 마스크 생성 함수 |
| Step 5 | 모델 초기화, 학습률 스케줄러, 학습 루프 |
| Step 6 | 평가 및 어텐션 시각화 |


---


---
## Step 1. 환경 준비 및 라이브러리

학습에 필요한 라이브러리를 불러옵니다.

- `torch`: 신경망 모델을 만들고 학습시키는 핵심 라이브러리
- `numpy`: 위치 인코딩 숫자 테이블을 미리 계산할 때 사용
- `matplotlib`: 어텐션 가중치를 색상 지도(히트맵)로 그릴 때 사용
- `re`: 텍스트에서 불필요한 특수문자 등을 제거하는 정규식 도구


In [32]:
import torch
import numpy
import matplotlib
import re

print(torch.__version__)
print(numpy.__version__)
print(matplotlib.__version__)

2.7.1+cu118
2.2.6
3.10.3


In [33]:
import numpy as np

---
## Step 2. 데이터 정제 및 토큰화

### 2-1. 데이터 로드 및 중복 제거

`korean-english-park` 병렬 코퍼스(한국어-영어 문장 쌍 모음)를 불러옵니다.
한국어 문장과 영어 문장을 탭(`\t`)으로 연결한 뒤, `set()`을 사용해 **완전히 동일한 중복 문장 쌍을 제거**합니다.


In [34]:
import os

data_dir = os.path.join(os.getenv("HOME"), 'work/transformer/data')
kor_path = data_dir + "/korean-english-park.train.ko"
eng_path = data_dir + "/korean-english-park.train.en"

def clean_corpus(kor_path, eng_path):
    with open(kor_path, "r") as f:
        kor = f.read().splitlines()

    with open(eng_path, "r") as f:
        eng = f.read().splitlines()

    assert len(kor) == len(eng)

    cleaned_corpus = list(set([
        "\t".join([k, e]) for k, e in zip(kor, eng) #   - zip()으로 한/영 문장을 짝지은 뒤, "\t"(탭) 문자로 연결하여 하나의 문자열로 만듭니다.
    ])) #   - set() 자료형에 넣어 완전히 동일한 문장 쌍(중복 데이터)을 제거합니다.

    return cleaned_corpus #  - 정제된 말뭉치 획득: 중복이 제거된 "한국어문장\t영어문장" 형태의 리스트가 반환됩니다.

cleaned_corpus = clean_corpus(kor_path, eng_path)

### 2-2. 패키지 설치

- **sentencepiece**: 언어에 상관없이 단어를 작은 조각(서브워드)으로 쪼개는 토크나이저입니다. 사전에 없는 단어가 나와도 대응할 수 있습니다.
- **tqdm**: 학습 중 진행 상황을 막대로 보여주는 도구입니다.


In [35]:
!pip install -q sentencepiece tqdm

### 2-3. 텍스트 정규화

학습 전 텍스트를 깔끔하게 정리합니다.

1. 영어 대문자를 모두 소문자로 변환 (`lower()`)
2. 한국어·영어·구두점 이외의 불필요한 문자 제거
3. `.`, `!`, `?`, `,` 같은 구두점 앞뒤에 공백 추가 → 독립적인 단어처럼 처리
4. 연속된 공백 정리

이렇게 하면 모델이 노이즈 없이 깨끗한 문장을 학습할 수 있습니다.


In [36]:
def preprocess_sentence(sentence):

  sentence = sentence.lower() # 소문자 변환
  sentence = re.sub(r"[^a-zA-Z가-힣?.!,]+", " ", sentence) # 노이즈 제거
  sentence = re.sub(r"([?.!,])", r" \1 ", sentence) # 구두점 분리
  sentence = re.sub(r'[" "]+', " ", sentence) # 공백 정리

  return sentence

### 2-4. SentencePiece 토크나이저 학습

한국어와 영어 각각 **SentencePiece Unigram** 방식으로 토크나이저를 학습합니다.
각 언어마다 최대 20,000개의 토큰(단어 조각)을 갖도록 설정합니다.

아래 4개의 특수 토큰은 모델 학습에서 특별한 역할을 합니다.

| ID | 토큰 | 역할 |
|:---:|:---:|---|
| 0 | `<pad>` | 문장 길이를 맞추기 위한 빈 칸 채우기 |
| 1 | `<bos>` | 문장의 시작을 알리는 신호 |
| 2 | `<eos>` | 문장의 끝을 알리는 신호 |
| 3 | `<unk>` | 사전에 없는 토큰 대체용 |

영어 토크나이저는 문장을 인코딩할 때 자동으로 `<bos>`와 `<eos>`를 앞뒤에 붙입니다.


In [37]:
# Sentencepiece를 활용하여 학습한 tokenizer를 생성합니다.
def generate_tokenizer(corpus,
                        vocab_size,
                        lang="ko",
                        pad_id=0,
                        bos_id=1,
                        eos_id=2,
                        unk_id=3):
    file = "./%s_corpus.txt" % lang
    model = "%s_spm" % lang

    with open(file, 'w', encoding='utf-8') as f:
        for row in corpus:
            f.write(str(row) + '\n')

    import sentencepiece as spm

    # 실제 데이터 기준 최소 vocab 보정
    unique_chars = set("".join(corpus))
    min_vocab_size = len(unique_chars) + 10

    if vocab_size < min_vocab_size:
        vocab_size = min_vocab_size

    spm.SentencePieceTrainer.Train(
        f'--input={file} '
        f'--model_prefix={model} '
        f'--vocab_size={vocab_size} '
        f'--character_coverage=0.9995 '
        f'--model_type=unigram '
        f'--pad_id={pad_id} '
        f'--bos_id={bos_id} '
        f'--eos_id={eos_id} '
        f'--unk_id={unk_id}'
    )

    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load(f'{model}.model')

    return tokenizer


SRC_VOCAB_SIZE = TGT_VOCAB_SIZE = 20000 # 단어 사전 크기 정의 (각 언어당 20,000개의 토큰을 가질 수 있도록 설정)

eng_corpus = []
kor_corpus = []

for pair in cleaned_corpus: # 앞서 정제한 cleaned_corpus에서 한/영 문장을 분리하고 정규화(preprocess)를 수행합니다.
    k, e = pair.split("\t")

    kor_corpus.append(preprocess_sentence(k))
    eng_corpus.append(preprocess_sentence(e))

ko_tokenizer = generate_tokenizer(kor_corpus, SRC_VOCAB_SIZE, "ko") # 한국어와 영어 각각의 토크나이저를 학습 및 생성합니다.
en_tokenizer = generate_tokenizer(eng_corpus, TGT_VOCAB_SIZE, "en")
en_tokenizer.set_encode_extra_options("bos:eos") # 특수 옵션 설정: 인코딩 시 자동으로 문장 앞뒤에 BOS(1)와 EOS(2)를 붙이도록 설정합니다.


sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=./ko_corpus.txt --model_prefix=ko_spm --vocab_size=20000 --character_coverage=0.9995 --model_type=unigram --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./ko_corpus.txt
  input_format: 
  model_prefix: ko_spm
  model_type: UNIGRAM
  vocab_size: 20000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  har

True

### 2-5. 토큰화 + 패딩

- 너무 긴 문장은 학습 효율을 떨어뜨리므로, **토큰 길이가 50 이하인 문장만** 남깁니다.
- 배치 내 문장들은 길이가 제각각이므로, 짧은 문장 뒤에 `0(<pad>)`을 붙여 **길이를 통일(패딩)**합니다.

최종적으로 만들어지는 데이터:
- `enc_train`: 인코더 입력 — 한국어 문장을 토큰 ID로 변환한 행렬
- `dec_train`: 디코더 입력 — 영어 문장을 토큰 ID로 변환한 행렬 (BOS/EOS 포함)


In [38]:
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm  # 진행 과정 보기

src_corpus = []
tgt_corpus = []

assert len(kor_corpus) == len(eng_corpus)

# 토큰의 길이가 50 이하인 문장만 남깁니다.
for idx in tqdm(range(len(kor_corpus))):
    src_tokens = ko_tokenizer.encode_as_ids(kor_corpus[idx])
    tgt_tokens = en_tokenizer.encode_as_ids(eng_corpus[idx])

    if len(src_tokens) <= 50 and len(tgt_tokens) <= 50:
        src_corpus.append(torch.tensor(src_tokens, dtype=torch.long))
        tgt_corpus.append(torch.tensor(tgt_tokens, dtype=torch.long))

def pad_sequences(sequences, padding_value=0):
    return torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=padding_value)

# 패딩처리를 완료하여 학습용 데이터를 완성합니다.
enc_train = pad_sequences(src_corpus, padding_value=0)
dec_train = pad_sequences(tgt_corpus, padding_value=0)

print(enc_train.shape, dec_train.shape)

  0%|          | 0/78968 [00:00<?, ?it/s]

torch.Size([72107, 50]) torch.Size([72107, 50])


---
## Step 3. 모델 아키텍처

아래는 이 노트북에서 구현할 Transformer 전체 흐름입니다.
한국어 토큰이 들어와서 영어 번역이 출력되기까지의 과정을 단계별로 나타냅니다.

```
입력 토큰 ID
    │
    ▼
[Embedding × √d_model + Positional Encoding]  ←── 위치 정보 주입
    │
    ▼  × N_layers
[Encoder Layer]
  ├─ Multi-Head Self-Attention (src_pad_mask)
  ├─ Add & LayerNorm  (Pre-LN)
  ├─ Position-wise FFN
  └─ Add & LayerNorm

    │ (encoder output)
    ▼
[Decoder Layer]  ×  N_layers  (디코더 입력 동시 처리)
  ├─ Masked Multi-Head Self-Attention (dec_mask = pad | causal)
  ├─ Add & LayerNorm
  ├─ Multi-Head Cross-Attention (Q=dec, K=V=enc_out, enc_dec_mask)
  ├─ Add & LayerNorm
  ├─ Position-wise FFN
  └─ Add & LayerNorm

    │
    ▼
[Linear → Logits (vocab_size)]
    │
    ▼
Cross-Entropy Loss  /  Greedy Argmax (추론)
```

> 위 구조가 각 단계에서 코드로 어떻게 구현되는지 아래에서 확인한다.


In [39]:
# ※ 아래는 초기 스케치 버전입니다.
# 실제 사용되는 완성된 Transformer 클래스는 아래 셀(Cell 20)에 정의되어 있습니다.
# (Encoder/Decoder 분리, Positional Encoding, 마스크 처리 포함)

import torch.nn as nn  # nn은 이후 셀에서도 사용됩니다.

---
### 3-1. 위치 인코딩 (Positional Encoding)

어텐션 메커니즘은 문장에서 단어 순서를 자동으로 파악하지 못합니다.
그래서 **각 단어의 위치 정보를 별도로 만들어 임베딩에 더해줍니다.**

- 짝수 번째 차원에는 **sin** 값, 홀수 번째 차원에는 **cos** 값을 사용합니다.
- 위치가 다르면 항상 다른 패턴이 만들어져서, 모델이 단어 위치를 구분할 수 있습니다.
- 학습 파라미터가 아니라 NumPy로 미리 고정값을 계산해 두고, 나중에 임베딩 벡터에 더합니다.

`Transformer` 클래스 안에서 `self.pos_encoding`으로 저장되며,
`embedding()` 메서드에서 시퀀스 길이만큼 잘라 임베딩에 더하는 방식으로 사용됩니다.


In [40]:
def positional_encoding(pos, d_model):
    def cal_angle(position, i):

        return position / np.power(10000, int(i) / d_model)

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table

---
### 3-2. Multi-Head Attention

<div align="center">
<img src="https://upload.wikimedia.org/wikipedia/commons/1/1b/Transformer%2C_attention_block_diagram.png"
     alt="Scaled Dot-Product Attention" width="200"
     style="background-color:white; padding:10px; border-radius:6px;"/>
&nbsp;&nbsp;
<img src="https://upload.wikimedia.org/wikipedia/commons/d/d2/Multiheaded_attention%2C_block_diagram.png"
     alt="Multi-Head Attention" width="260"
     style="background-color:white; padding:10px; border-radius:6px;"/>
</div>

#### Scaled Dot-Product Attention

어텐션의 핵심 아이디어는 **"어떤 단어가 다른 단어와 얼마나 관련이 있는가"** 를 숫자로 표현하는 것입니다.

| 단계 | 하는 일 | 이유 |
|---|---|---|
| Q × K의 전치 | 쿼리-키 유사도 점수 계산 | 같은 방향일수록 점수가 높음 |
| √d_k 나누기 | 점수 크기 조정 | 점수가 너무 크면 softmax가 한쪽으로 쏠림 |
| 마스킹 | 봐서는 안 될 위치를 매우 작은 값으로 채움 | softmax 후 해당 위치의 영향이 0에 가까워짐 |
| softmax | 모든 점수를 합이 1인 확률로 변환 | 각 위치에 "얼마나 집중할지" 비율이 됨 |
| × V | 확률 비율로 V를 가중 합산 | 중요한 단어 정보를 더 많이 가져옴 |

#### Multi-Head 흐름

입력을 여러 개의 "헤드"로 나눠서 각 헤드가 **서로 다른 관점**으로 어텐션을 수행한 뒤 합칩니다.
예를 들어 한 헤드는 문법적 관계, 다른 헤드는 의미적 관계에 집중할 수 있습니다.

```
입력 Q/K/V: (B, seq, d_model)
      │
      ▼  W_q, W_k, W_v (선형 변환)
(B, seq, d_model)
      │
      ▼  split_heads → 헤드별로 분리
(B, n_heads, seq, depth)
      │
      ▼  헤드별 어텐션 계산
(B, n_heads, seq, depth)
      │
      ▼  combine_heads → 다시 합치기
(B, seq, d_model)
      │
      ▼  linear (최종 선형 변환)
(B, seq, d_model)
```

> **마스크 규약**: 이 구현에서 `mask == 0`인 자리를 차단합니다. 즉 `1.0 = 볼 수 있음`, `0.0 = 가림`입니다.


In [41]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        d_k = K.shape[-1]
        QK = torch.matmul(Q, K.transpose(-2, -1))
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        if mask is not None:
            scaled_qk = scaled_qk.masked_fill(mask == 0, float('-1e9'))

        attentions = F.softmax(scaled_qk, dim=-1)
        out = torch.matmul(attentions, V)

        return out, attentions

    def split_heads(self, x):
        bsz, seq_len, d_model = x.shape
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        x = x.permute(0, 2, 1, 3)
        return x

    def combine_heads(self, x):
        bsz, num_heads, seq_len, depth = x.shape
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        WQ = self.W_q(Q)
        WK = self.W_k(K)
        WV = self.W_v(V)

        WQ_splits = self.split_heads(WQ)
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask)

        out = self.combine_heads(out)
        out = self.linear(out)

        return out, attention_weights

---
### 3-3. Position-wise Feed-Forward Network (FFN)

어텐션이 "어느 단어에 집중할지" 결정한다면,
FFN은 그 결과를 받아 **실제 의미적 변환을 수행**합니다.

- 구조: `d_model → d_ff → d_model` (중간에서 차원을 크게 늘렸다가 다시 줄임)
- 중간에 **ReLU** 활성화 함수를 넣어 표현력을 높입니다.
- 각 위치(토큰)에 **동일한 변환을 독립적으로** 적용합니다.
- 이 노트북에서는 `d_model=512`, `d_ff=2048`로 설정합니다 (약 4배 확장).


In [42]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)

        return out

---
### 3-4. Encoder Layer

인코더 한 층이 하는 일을 순서대로 보면 다음과 같습니다.

```
입력 x: (B, src_len, d_model)
  │
  ├── ① LayerNorm으로 입력 정규화
  │   ② Self-Attention: 문장 내 단어들끼리 서로 참조
  │   ③ 잔차 연결(Residual): 원래 값에 더하기 → 학습 안정화
  │
  ├── ④ LayerNorm으로 정규화
  │   ⑤ FFN: 각 위치 개별 변환
  │   ⑥ 잔차 연결
  │
  └── 출력: (B, src_len, d_model) ← 입력과 같은 크기
```

**중요 포인트:**
- Self-Attention이므로 Q, K, V 모두 같은 문장에서 나옵니다. 즉 **한국어 문장 내부의 단어들이 서로를 참조**합니다.
- 인코더는 문장 전체를 **양방향으로** 읽습니다. (미래 단어도 볼 수 있음)
- 단, `<pad>`로 채워진 위치는 어텐션에서 무시합니다.


In [43]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()

        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        """
        Multi-Head Attention
        """
        residual = x
        out = self.norm_1(x)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.dropout(out)
        out += residual

        """
        Position-Wise Feed Forward Network
        """
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.dropout(out)
        out += residual

        return out, enc_attn

---
### 3-5. Decoder Layer

<div align="center">
<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/Encoder_cross-attention%2C_multiheaded_version.png/500px-Encoder_cross-attention%2C_multiheaded_version.png"
     alt="Cross-attention" width="320"
     style="background-color:white; padding:12px; border-radius:6px;"/>
</div>

디코더 한 층은 세 단계로 구성됩니다.

```
입력 y: (B, tgt_len, d_model)     enc_out: (B, src_len, d_model)
  │
  ├── [1] Masked Self-Attention
  │       영어 문장(타깃) 내부 단어들이 서로 참조
  │       단, 아직 생성되지 않은 미래 단어와 <pad>는 가림
  │
  ├── [2] Cross-Attention
  │       디코더가 인코더 출력(한국어 정보)을 참조
  │       "지금 어떤 영어 단어를 생성할지 결정하기 위해 한국어 문장의 어느 부분을 봐야 하는가"
  │
  └── [3] FFN: 최종 의미 변환
```

**세 마스크의 역할 비교**

| 마스크 | 차단 대상 | 사용 위치 |
|---|---|---|
| `enc_mask` | 한국어 `<pad>` | 인코더 self-attention |
| `dec_mask` | 영어 `<pad>` + 미래 토큰 | 디코더 self-attention |
| `enc_dec_mask` | 한국어 `<pad>` | Cross-attention |


In [44]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()

        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)

        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, dec_mask, enc_dec_mask):
        """
        Masked Multi-Head Attention
        """
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, dec_mask)
        out = self.dropout(out)
        out += residual

        """
        Multi-Head Attention
        """
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, enc_dec_mask)
        out = self.dropout(out)
        out += residual

        """
        Position-Wise Feed Forward Network
        """
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.dropout(out)
        out += residual

        return out, dec_attn, dec_enc_attn

---
### 3-6. Encoder 스택

인코더 레이어를 `n_layers`개 쌓습니다.
앞 층의 출력이 다음 층의 입력이 되며, 같은 마스크가 모든 층에 전달됩니다.
층이 깊어질수록 더 추상적인 언어 표현을 학습합니다.

```
x: (B, src_len, d_model)   ──┐
                              │  × n_layers
  ┌───────────────────────────┘
  │ EncoderLayer  →  (B, src_len, d_model)
  └───────────────────────────┐
                              │
enc_out: (B, src_len, d_model)
```


In [45]:
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.n_layers = n_layers
        self.enc_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        out = x
        enc_attns = []

        for i in range(self.n_layers):
            out, enc_attn = self.enc_layers[i](out, mask)
            enc_attns.append(enc_attn)

        return out, enc_attns

---
### 3-7. Decoder 스택

디코더 레이어를 `n_layers`개 쌓습니다.
모든 디코더 층은 **동일한 인코더 출력(`enc_out`)**을 cross-attention에서 참조합니다.
즉, 모든 디코더 층이 한국어 문장 정보를 직접 참고합니다.


In [46]:
class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

    def forward(self, x, enc_out, dec_mask, enc_dec_mask):
        out = x

        dec_attns = []
        dec_enc_attns = []

        for i in range(self.n_layers):
            out, dec_attn, dec_enc_attn = self.dec_layers[i](
                out, enc_out, dec_mask, enc_dec_mask
            )

            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)

        return out, dec_attns, dec_enc_attns

---
### 3-8. Transformer 전체 조립

```
한국어 입력 (enc_in)  ──▶  Encoder  ──▶  enc_out ──────────────────────┐
                                                                        ▼
영어 입력 (dec_in)    ──▶  Decoder (enc_out 참조)  ──▶  dec_out  ──▶  Linear  ──▶  logits
```

**weight sharing (가중치 공유, `shared=True`)**:
디코더 입력 임베딩과 출력 선형 변환이 **같은 가중치**를 씁니다.
같은 단어가 입력에 들어올 때와 출력으로 나올 때 동일한 벡터 표현을 사용하므로
파라미터 수를 줄이고 학습 효율을 높입니다.

**임베딩 스케일링**:
임베딩 벡터에 `√d_model`을 곱합니다.
이렇게 하면 임베딩 값의 크기가 위치 인코딩 값과 비슷해져서,
두 값을 더했을 때 한쪽이 다른 쪽을 압도하지 않습니다.


In [47]:
class Transformer(nn.Module):
    def __init__(self,
                 n_layers,
                 d_model,
                 n_heads,
                 d_ff,
                 src_vocab_size,
                 tgt_vocab_size,
                 pos_len,
                 dropout=0.2,
                 shared=True):

        super(Transformer, self).__init__()

        self.d_model = d_model
        self.shared = shared

        self.enc_emb = nn.Embedding(src_vocab_size, d_model)
        self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        self.pos_encoding = torch.tensor(
            positional_encoding(pos_len, d_model),
            dtype=torch.float32
        )
        self.dropout = nn.Dropout(dropout)

        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        self.fc = nn.Linear(d_model, tgt_vocab_size)

        if shared:
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        seq_len = x.shape[1]

        out = emb(x)

        if self.shared:
            out *= torch.sqrt(torch.tensor(self.d_model, dtype=torch.float32))

        pos_enc = self.pos_encoding[:seq_len, :].unsqueeze(0).to(x.device)
        out += pos_enc

        out = self.dropout(out)
        return out

    def forward(self, enc_in, dec_in, enc_mask, causality_mask, dec_mask):
        enc_in = self.embedding(self.enc_emb, enc_in)
        dec_in = self.embedding(self.dec_emb, dec_in)

        enc_out, enc_attns = self.encoder(enc_in, enc_mask)

        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in, enc_out, dec_mask, causality_mask)

        logits = self.fc(dec_out)

        return logits, enc_attns, dec_attns, dec_enc_attns


---
## Step 4. 마스크 생성 함수

마스크는 어텐션에서 **특정 위치를 보지 못하도록 가리는 역할**을 합니다.
이 구현에서는 **`1.0 = 가림`**, `0.0 = 볼 수 있음` 규약을 사용합니다.
가려진 자리에는 매우 작은 값(-1e9)을 채워 softmax 후 어텐션 가중치가 0에 가까워지게 합니다.

### 4-1. Padding Mask
`<pad>` 토큰(ID=0)인 위치를 1.0으로 만들어 어텐션에서 무시합니다.
실제 단어가 아닌 자리를 읽지 않도록 막는 것입니다.

### 4-2. Causal (Look-ahead) Mask
디코더가 **아직 생성하지 않은 미래 단어를 미리 보지 못하도록** 막습니다.
예를 들어 2번째 단어를 예측할 때는 0, 1번째 단어만 볼 수 있습니다.

```
위치 0 → 자기 자신만 볼 수 있음
위치 1 → 0, 1번 위치를 볼 수 있음
위치 2 → 0, 1, 2번 위치를 볼 수 있음
...
```

### 4-3. Decoder Self-Attention Mask (통합)
패딩 마스크와 causal 마스크를 `torch.max`로 합칩니다.
두 마스크 중 하나라도 "가림"이면 해당 위치가 가려집니다.

### 4-4. 마스크 타입별 요약

| 마스크 | 차단 대상 | 사용 위치 |
|---|---|---|
| `enc_mask` | source `<pad>` | Encoder self-attention |
| `dec_mask` | target `<pad>` + 미래 토큰 | Decoder self-attention |
| `enc_dec_mask` | source `<pad>` | Cross-attention |


In [48]:
def generate_padding_mask(seq):
    """ 패딩된 부분(0)을 1로 변환하여 마스크 생성 """
    mask = (seq == 0).float()
    return mask[:, None, None, :]

def generate_causality_mask(src_len, tgt_len):
    """ 미래 정보를 참조하지 않도록 Causal Mask 생성 """
    mask = 1 - torch.cumsum(torch.eye(src_len, tgt_len), dim=0)
    return mask.float()

def generate_masks(src, tgt):
    """ Encoder-Decoder에서 사용할 마스크 생성 """
    enc_mask = generate_padding_mask(src)
    dec_mask = generate_padding_mask(tgt)

    dec_causality_mask = generate_causality_mask(tgt.shape[1], tgt.shape[1])
    dec_mask = torch.max(dec_mask, dec_causality_mask.to(dec_mask.device))

    dec_enc_causality_mask = generate_causality_mask(tgt.shape[1], src.shape[1])
    dec_enc_mask = torch.max(enc_mask, dec_enc_causality_mask.to(enc_mask.device))

    return enc_mask, dec_enc_mask, dec_mask

---
## Step 5. 모델 초기화 및 학습 설정

### 5-1. 모델 인스턴스 생성

Transformer 모델을 아래 설정으로 만듭니다.

| 파라미터 | 값 | 설명 |
|---|---|---|
| `n_layers` | 2 | 인코더/디코더 각 2층 (원논문은 6층, 학습 시간 단축을 위해 축소) |
| `d_model` | 512 | 각 토큰을 표현하는 벡터 크기 |
| `n_heads` | 8 | 어텐션 헤드 수 (512 / 8 = 헤드당 64차원) |
| `d_ff` | 2048 | FFN 내부 차원 (d_model의 4배) |
| `pos_len` | 200 | 최대 200 토큰까지 위치 인코딩 지원 |
| `dropout` | 0.2 | 과적합 방지를 위해 학습 중 20% 뉴런 랜덤 비활성화 |


In [49]:
transformer = Transformer(
    n_layers=2,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    pos_len=200,
    dropout=0.2,
    shared=True
)

---
### 5-2. Warmup 학습률 스케줄러

학습 초반에는 학습률을 **서서히 높이다가(워밍업)**, 이후에는 **점점 낮춥니다.**

- **워밍업 구간** (step < 4000): 학습률이 선형으로 증가합니다. 초반 불안정한 학습을 방지합니다.
- **감소 구간** (step ≥ 4000): 학습률이 step에 반비례해 점차 줄어듭니다.

```
학습률
 ▲
 │      ╭──────  (이후 점점 감소)
 │    ╭─╯
 │  ╭─╯
 │╭─╯
 └────────────────────── step
   0    4000
   (워밍업)
```


In [50]:
import math

class LearningRateScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, d_model, warmup_steps=4000, last_epoch=-1):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        super(LearningRateScheduler, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        step = max(1, self.last_epoch)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        lr = (self.d_model ** -0.5) * min(arg1, arg2)
        return [lr for _ in self.base_lrs]

---
### 5-3. 옵티마이저

**Adam 옵티마이저**를 사용합니다.
초기 학습률은 스케줄러가 step에 따라 자동으로 조정합니다.


In [51]:
optimizer = torch.optim.Adam(
    transformer.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)

learning_rate = LearningRateScheduler(
    optimizer,
    d_model=512
)


---
### 5-4. 학습 루프

**Teacher Forcing**: 학습 중 디코더에 실제 정답 문장을 한 칸씩 밀어 넣어 줍니다.
모델이 틀린 예측을 해도 다음 입력에는 정답을 주기 때문에 학습이 안정적으로 진행됩니다.

```
원본 타깃:   [<bos>, tok1, tok2, tok3, <eos>]
디코더 입력: [<bos>, tok1, tok2, tok3]       ← 마지막 토큰 제거
정답 레이블: [tok1,  tok2, tok3, <eos>]      ← 첫 <bos> 제거
              각 위치에서 "다음 토큰이 무엇인가"를 예측
```

**Gradient Clipping**: 기울기(gradient)가 너무 커지면 학습이 폭발할 수 있습니다.
기울기 크기를 1.0 이하로 잘라서 학습을 안정화합니다.

**체크포인트**: 매 에포크마다 `./checkpoints/` 폴더에 모델 상태를 저장합니다.
학습이 중단되어도 이어서 재개할 수 있습니다.

**번역 추론 (`evaluate()`)**: 학습 후 실제 번역을 수행할 때는
`<bos>` 한 토큰에서 시작해서, 매 스텝마다 가장 확률이 높은 다음 토큰을 하나씩 선택합니다.
`<eos>`가 나오거나 최대 길이(50)에 도달하면 번역을 종료합니다.


In [52]:
# BOS/EOS 토큰 ID
TGT_BOS = en_tokenizer.bos_id()  # 1
TGT_EOS = en_tokenizer.eos_id()  # 2

# ================================
# 5️⃣ Transformer 학습 진행
# ================================
import os
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformer = transformer.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)

# ── 하이퍼파라미터 ───────────────────────────────
EPOCHS     = 10
BATCH_SIZE = 64        # ← 배치 처리로 에포크 속도 대폭 향상
CLIP       = 1.0       # gradient clipping
CKPT_DIR   = "./checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# ── DataLoader ───────────────────────────────────
dataset    = TensorDataset(enc_train, dec_train)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

sample_sentence = "안녕하세요."

# ── 마스크를 배치 단위로 생성 ────────────────────
def generate_masks_batch(src, tgt):
    """배치(B, src_len / tgt_len) 입력에 대해 3종류 마스크 생성"""
    B, src_len = src.shape
    tgt_len    = tgt.shape[1]

    # Encoder padding mask: (B, 1, 1, src_len)
    enc_mask = (src == 0).float().unsqueeze(1).unsqueeze(2)

    # Decoder padding mask: (B, 1, 1, tgt_len)
    dec_pad  = (tgt == 0).float().unsqueeze(1).unsqueeze(2)

    # Causal mask: (1, 1, tgt_len, tgt_len)
    causal   = (1 - torch.tril(torch.ones(tgt_len, tgt_len))).unsqueeze(0).unsqueeze(0)

    # Decoder self-attn mask = padding ∪ causal: (B, 1, tgt_len, tgt_len)
    dec_mask = torch.max(dec_pad, causal.to(dec_pad.device))

    # Cross-attn mask: (B, 1, 1, src_len) — enc padding만 적용
    enc_dec_mask = enc_mask

    return enc_mask, enc_dec_mask, dec_mask

# ── Evaluate ─────────────────────────────────────
def evaluate(sentence):
    transformer.eval()
    sentence = preprocess_sentence(sentence)
    tokens   = ko_tokenizer.encode_as_ids(sentence)
    tokens   = torch.tensor(tokens).unsqueeze(0).to(device)
    output   = torch.tensor([[TGT_BOS]], device=device)

    for _ in range(50):
        enc_mask, enc_dec_mask, dec_mask = generate_masks_batch(tokens, output)
        enc_mask     = enc_mask.to(device)
        enc_dec_mask = enc_dec_mask.to(device)
        dec_mask     = dec_mask.to(device)

        with torch.no_grad():
            predictions, *_ = transformer(
                tokens, output, enc_mask, enc_dec_mask, dec_mask
            )

        predicted_id = torch.argmax(predictions[:, -1:, :], dim=-1)
        if predicted_id.item() == TGT_EOS:
            break
        output = torch.cat([output, predicted_id], dim=-1)

    return en_tokenizer.decode_ids(output.squeeze().cpu().numpy().tolist())

# ── 체크포인트 저장 / 불러오기 ───────────────────
def save_checkpoint(epoch, avg_loss):
    path = os.path.join(CKPT_DIR, f"ckpt_epoch{epoch+1:02d}.pt")
    torch.save({
        "epoch":                epoch + 1,
        "model_state_dict":     transformer.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": learning_rate.state_dict(),
        "avg_loss":             avg_loss,
    }, path)
    print(f"  💾 체크포인트 저장: {path}")

def load_checkpoint(path):
    ckpt = torch.load(path, map_location=device)
    transformer.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    learning_rate.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"]
    print(f"  ✅ 체크포인트 로드: epoch {start_epoch}부터 재개")
    return start_epoch

# ── 이어서 학습하려면 아래 주석 해제 ─────────────
# START_EPOCH = load_checkpoint("./checkpoints/ckpt_epoch05.pt")
START_EPOCH = 0

# ── 학습 루프 ─────────────────────────────────────
for epoch in range(START_EPOCH, EPOCHS):

    transformer.train()
    total_loss = 0

    for step, (src, tgt) in enumerate(dataloader):
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input  = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        enc_mask, enc_dec_mask, dec_mask = generate_masks_batch(src, tgt_input)
        enc_mask     = enc_mask.to(device)
        enc_dec_mask = enc_dec_mask.to(device)
        dec_mask     = dec_mask.to(device)

        optimizer.zero_grad()

        predictions, *_ = transformer(
            src, tgt_input, enc_mask, enc_dec_mask, dec_mask
        )

        predictions = predictions.reshape(-1, predictions.size(-1))
        tgt_output  = tgt_output.reshape(-1)

        loss = criterion(predictions, tgt_output)
        loss.backward()

        # Gradient clipping — 학습 안정화
        torch.nn.utils.clip_grad_norm_(transformer.parameters(), CLIP)

        optimizer.step()
        learning_rate.step()

        total_loss += loss.item()

        if step % 50 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | Step {step}/{len(dataloader)} | Loss {loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)

    print(f"\n🌌 Epoch {epoch+1} 완료 | Average Loss: {avg_loss:.4f}")

    translated = evaluate(sample_sentence)
    print(f"입력: {sample_sentence}  →  번역: {translated}")

    # 매 에포크마다 체크포인트 저장
    save_checkpoint(epoch, avg_loss)
    print("=" * 50)

print("✨ 학습 종료")


Epoch 1/10 | Step 0/1127 | Loss 11635.7559
Epoch 1/10 | Step 50/1127 | Loss 11510.6279
Epoch 1/10 | Step 100/1127 | Loss 11058.7041
Epoch 1/10 | Step 150/1127 | Loss 10888.2041
Epoch 1/10 | Step 200/1127 | Loss 10339.9990
Epoch 1/10 | Step 250/1127 | Loss 9566.1787
Epoch 1/10 | Step 300/1127 | Loss 8852.1201
Epoch 1/10 | Step 350/1127 | Loss 7784.0674
Epoch 1/10 | Step 400/1127 | Loss 6664.6567
Epoch 1/10 | Step 450/1127 | Loss 6013.7480
Epoch 1/10 | Step 500/1127 | Loss 5295.3403
Epoch 1/10 | Step 550/1127 | Loss 4785.0654
Epoch 1/10 | Step 600/1127 | Loss 4469.8608
Epoch 1/10 | Step 650/1127 | Loss 4141.5288
Epoch 1/10 | Step 700/1127 | Loss 3968.4749
Epoch 1/10 | Step 750/1127 | Loss 3590.5940
Epoch 1/10 | Step 800/1127 | Loss 3710.5894
Epoch 1/10 | Step 850/1127 | Loss 3470.3513
Epoch 1/10 | Step 900/1127 | Loss 3249.6643
Epoch 1/10 | Step 950/1127 | Loss 3248.6467
Epoch 1/10 | Step 1000/1127 | Loss 3051.5078
Epoch 1/10 | Step 1050/1127 | Loss 3100.2705
Epoch 1/10 | Step 1100/1127 

## Epoch 10
#### Loss ≈ 11.69
#### 반복 출력 발생
#### 의미 번역 실패

---
## 정리

구현한 Transformer의 핵심 구성 요소와 코드 대응을 정리합니다.

| 구성 요소 | 핵심 구현 | 위치 |
|---|---|---|
| 토큰 임베딩 | `nn.Embedding(vocab_size, d_model)` | `Transformer.embedding()` |
| 위치 인코딩 | sin/cos 테이블을 미리 계산해 임베딩에 더함 | `positional_encoding()`, `Transformer.embedding()` |
| 임베딩 스케일링 | 임베딩 벡터에 `√d_model`을 곱해 위치 인코딩과 크기를 맞춤 | `Transformer.embedding()` |
| Scaled Dot-Product Attention | Q와 K의 유사도 계산 후 softmax, V 가중합 | `MultiHeadAttention.scaled_dot_product_attention()` |
| Multi-Head 분할/병합 | 헤드별로 나눠 병렬 어텐션 후 다시 합침 | `MultiHeadAttention` |
| Position-wise FFN | `Linear → ReLU → Linear` | `PoswiseFeedForwardNet` |
| Pre-LN 잔차 연결 | 서브레이어 앞에 LayerNorm 적용 후 잔차 더하기 | `EncoderLayer`, `DecoderLayer` |
| Padding Mask | `<pad>` 위치(ID=0)를 1.0으로 표시해 어텐션 차단 | `generate_padding_mask()` |
| Causal Mask | 미래 위치를 상삼각 행렬로 차단 | `generate_causality_mask()` |
| Decoder Self Mask | 패딩 마스크와 causal 마스크를 합침 | `generate_masks_batch()` |
| Weight Sharing | 디코더 임베딩과 출력 선형층의 가중치를 공유 | `Transformer.__init__()` |
| Warmup 학습률 | 초반 선형 증가 후 점차 감소 | `LearningRateScheduler` |
| Teacher Forcing | 학습 시 정답 타깃을 한 칸씩 밀어 디코더에 입력 | 학습 루프 |
| 자기회귀 추론 | `<bos>`부터 시작해 argmax로 한 토큰씩 생성 | `evaluate()` |
| Gradient Clipping | 기울기 크기를 1.0 이하로 잘라 학습 안정화 | 학습 루프 |
| Checkpoint | 에포크마다 모델/옵티마이저 상태 저장 및 로드 | `save_checkpoint()` / `load_checkpoint()` |


### 후속 학습 방향

- **인코더 전용** (BERT 계열): 디코더 없이 양방향 어텐션으로 문장 이해에 특화
- **디코더 전용** (GPT 계열): 크로스 어텐션 없이 causal mask만으로 텍스트 생성
- **위치 표현 변형**: 학습 가능한 위치 임베딩, RoPE, ALiBi 등 다양한 방식
- **효율적 어텐션**: FlashAttention, Sparse Attention으로 메모리/속도 개선

